## 音频分割准备

准备工作有：
1. 从传入的path中取出所有的文件列表
2. 判断传递的voice文件是否存在
3. 加载音频文件到numpy

音频分割所需要的参数：
- `source`: 音频文件的列表，默认：`./source/voices`
  > 源目录，传递的可能是单个文件，也可能是个目录。在这个目录中我们得到一个`voice`音频的文件列表。单个文件也转换为列表
  
- `output_root`: 输出目录的路径: 默认：`./output/slice_voices`

In [1]:
import os
import sys
import traceback

import numpy as np
import ffmpeg

### 1. 获取文件列表

In [2]:
# 把非代码相关的目录全部放这里去，这样方便后续清理

source = "../../../data/audio/moda"
target_root = "../../../output/slice_test"

In [3]:
!ls {source}

asr_example.wav vad_example.wav


In [4]:
# 判断source是否是文件
os.path.isfile(source)

False

In [5]:
# 判断source是否是目录
os.path.isdir(source)

True

In [6]:
# 如果是目录，那么列出其里面的子文件
if os.path.isdir(source):
    files = list(os.listdir(source))
    print("files:", files)
else:
    print("不是目录")

files: ['asr_example.wav', 'vad_example.wav']


**把上面的代码整合到一起：** 
我们编写了一个`list_all_files`的函数，参数：
- `path`: 从哪个路径开始检查文件
- `list_sub_dir`: 是否列出子目录中的文件
- `ignore_hidden`: 忽略隐藏的文件，这里以`.`开头的文件就是隐藏文件

In [7]:
def list_all_files(path, list_sub_dir=True, ignore_hidden=True, use_abspath=True):
    # 1. 准备一个数组来存储文件列表
    files = []

    # 2. 先判断文件类型
    # 2-1. 文件就直接加入到files中
    if os.path.isfile(path):
        files = [path]
    # 2-2. 是目录，就需要检查所有的子文件/目录
    elif os.path.isdir(path):
        # 接下来遍历获取所有子文件
        for sub in os.listdir(path):
            # 是否忽略隐藏文件
            if ignore_hidden:
                if sub.startswith("."):
                    continue
            # 记得组装一下子文件的路径
            sub_path = os.path.join(path, sub)
            # 如果是目录，需要列出子目录中的文件，就递归调用自身
            if list_sub_dir and os.path.isdir(sub_path):
                sub_files = list_all_files(sub_path)
                # 把子目录中的文件添加到files中
                files.extend(sub_files)
            elif os.path.isfile(sub_path):
                files.append(sub_path)

    # 3. 返回所有的文件: 且排序一下
    if use_abspath:
        files = [os.path.abspath(p) for p in files]
    files.sort()
    return files

In [8]:
# 现在我们来测试一下看文件是否OK
files = list_all_files(source, use_abspath=False)
files

['../../../data/audio/moda/asr_example.wav',
 '../../../data/audio/moda/vad_example.wav']

### 2. 判断文件是否存在

In [9]:
files = list_all_files(source, use_abspath=False)
files

['../../../data/audio/moda/asr_example.wav',
 '../../../data/audio/moda/vad_example.wav']

In [10]:
for f in files:
    print(os.path.exists(f))

True
True


### 3. 加载音频文件到numpy

#### 3.1 使用ffmpeg

In [11]:
f

'../../../data/audio/moda/vad_example.wav'

In [12]:
# 采用ffmpeg读取音频文件
out, info = ffmpeg.input(f, threads=0)\
    .output("-", format="f32le", acodec="pcm_f32le", ac=1, ar=3200)\
    .run(cmd=["ffmpeg", "-nostdin"], capture_stdout=True, capture_stderr=True)

- input方法：
  - 指定输入文件为f
  - 线程数为threads=0：表示让`FFmpeg`自动选择适当的线程数
- output方法
  - "-": 这里是输出文件名，"-"表示输出到标准输出
  - format="f32le": 指定输出格式为32位浮点小端(little-endian)
  - acodec="pcm_f32le": 指定音频编码器为32位浮点小端PCM编码
  - ac=1：指定输出音频通道数为单声道（1个声道）
  - ar=3200: 指定输出音频的采样率
- run方法：
  - `cmd=["ffmpeg", "-nostdin"]`指定运行FFmpeg命令，使用-nostdin选项，意味着FFmpeg不会从标准输入读命令，这在脚本运行的时候避免意外的用户输入
  - `campture_stdout=True`: 捕获标准输出(stdout)，这意味着FFmpeg的输出捕获到变量而不是直接打印到控制台

In [13]:
type(out), len(out)

(bytes, 902024)

In [14]:
# 捕获到的stdout,stderr信息
print(info.decode("utf-8"))

ffmpeg version 4.4.4 Copyright (c) 2000-2023 the FFmpeg developers
  built with Apple clang version 15.0.0 (clang-1500.3.9.4)
  configuration: --prefix='/opt/homebrew/Cellar/ffmpeg@4/4.4.4_5' --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags= --enable-avresample --enable-ffplay --enable-gnutls --enable-gpl --enable-libaom --enable-libbluray --enable-libdav1d --enable-libmp3lame --enable-libopus --enable-librav1e --enable-librist --enable-librubberband --enable-libsnappy --enable-libsrt --enable-libtesseract --enable-libtheora --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libxvid --enable-lzma --enable-libfontconfig --enable-libfreetype --enable-frei0r --enable-libass --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-libopenjpeg --enable-libspeex --enable-libsoxr --enable-libzmq --enable-libzimg --disable-libjack --disable-indev=jack --enable-vide

In [15]:
# 把FFmpeg读取到的音频数据，转换为`np.float32`数组
np_audio = np.frombuffer(out, np.float32).flatten()
np_audio.shape

(225506,)

#### 3.2 使用librosa

In [16]:
import librosa

In [17]:
f

'../../../data/audio/moda/vad_example.wav'

In [18]:
audio_np, audio_sample_rate = librosa.load(f, sr=None, mono=False)
type(audio_np)

numpy.ndarray

In [19]:
audio_np.shape

(1127530,)

In [20]:
audio_sample_rate

16000